# IEM Cologne 2026 — Stage 2 Pick'Em Optimizer

Challenger Stage: 8 teams advancing from Stage 1 + 8 seeded teams entering fresh.

---

### Contents
1. [Load Artefacts](#1-load-artefacts)
2. [Stage 2 Team Configuration](#2-stage-2-team-configuration)
3. [Pick'Em Optimizer](#3-pickem-optimizer)
   - 3.1 Marginal Outcome Probabilities
   - 3.2 Optimal Pick Set
   - 3.3 Score Distribution
   - 3.4 Swiss Outcome Distribution
   - 3.5 Convergence Plot
   - 3.6 Pick Stability Heatmap

## 1. Load Artefacts

In [ ]:
import pickle
import sys, os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, os.getcwd())

from o1_config import DATA_DIR, MODEL_DIR, STAGE2_TEAMS
from o4_neural_net import MatchPredictor, pytorch_predict
from o7_simulate import precompute_win_probs, fetch_all_team_stats
from o8_pickem import run_simulations_chunked, optimize_from_histograms, score_dist_from_hist

feat_cols = pickle.load(open(MODEL_DIR / "feature_cols.pkl", "rb"))
scaler    = pickle.load(open(MODEL_DIR / "scaler.pkl",       "rb"))
elo       = pickle.load(open(MODEL_DIR / "elo.pkl",          "rb"))
blend_w   = pickle.load(open(MODEL_DIR / "blend_weight.pkl", "rb"))
input_dim = pickle.load(open(MODEL_DIR / "input_dim.pkl",    "rb"))

from xgboost import XGBClassifier
pt_model = MatchPredictor(input_dim)
pt_model.load_state_dict(torch.load(MODEL_DIR / "pytorch_model.pt", map_location="cpu"))
pt_model.eval()

xgb_model = XGBClassifier()
xgb_model.load_model(str(MODEL_DIR / "xgb_model.json"))

print(f"Blend: {blend_w:.0%} PyTorch + {1-blend_w:.0%} XGBoost")
print("Artefacts loaded.")

## 2. Stage 2 Team Configuration

**Fill in `STAGE1_ADVANCING` once all Stage 1 results are confirmed.**

Stage 2 = 8 teams advancing from Stage 1 + 8 seeded teams (Spirit, G2, Monte, paiN, Astralis, FUT, 9z, Legacy).

In [ ]:
# ⚠️  UPDATE THIS once all Stage 1 results are in
STAGE1_ADVANCING = [
    # "TeamA",
    # "TeamB",
    # "TeamC",
    # "TeamD",
    # "TeamE",
    # "TeamF",
    # "TeamG",
    # "TeamH",
]

# Stage 2 seeds (enter fresh from config)
STAGE2_SEEDS = list(STAGE2_TEAMS)

stage2_teams = STAGE1_ADVANCING + STAGE2_SEEDS

assert len(stage2_teams) == 16, (
    f"Expected 16 teams, got {len(stage2_teams)}. "
    "Fill in STAGE1_ADVANCING with all 8 teams that advanced."
)

print(f"Stage 2 teams ({len(stage2_teams)}):")
print(f"  From Stage 1 : {STAGE1_ADVANCING}")
print(f"  Seeded in    : {STAGE2_SEEDS}")

## 3. Pick'Em Optimizer

In [ ]:
PICKEM_N_SIMS = 5_000_000   # upper bound — convergence stops early when picks stabilise

# Fetch current team stats and compute pairwise win probabilities
team_dfs   = fetch_all_team_stats(teams=stage2_teams)
prob_cache = precompute_win_probs(
    stage2_teams, team_dfs, elo, pt_model, xgb_model,
    scaler, feat_cols, blend_w,
)

print(f"Running up to {PICKEM_N_SIMS:,} simulations...")
p30, padv, p03, combos, hists, conv_hist = run_simulations_chunked(
    stage2_teams, prob_cache, PICKEM_N_SIMS,
    converge=True, tol=0.001, patience=3,
)

best_picks, best_p5, best_idx = optimize_from_histograms(stage2_teams, combos, hists)
pick_dist = score_dist_from_hist(best_idx, hists)

print(f"\n=== STAGE 2 OPTIMAL PICK'EM  (n={int(hists[0].sum()):,}) ===")
print(f"  3-0  : {best_picks['three_zero']}")
print(f"  Adv  : {best_picks['advance']}")
print(f"  0-3  : {best_picks['zero_three']}")
print(f"  P(>=5 correct): {sum(v for k,v in pick_dist.items() if k>=5):.1%}")

In [ ]:
# 3.1 Marginal outcome probabilities
x     = np.arange(len(stage2_teams))
width = 0.28

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - width, p30,  width, label="P(3-0)",    color="#4CAF50", alpha=0.9)
ax.bar(x,         padv, width, label="P(advance)", color="#2196F3", alpha=0.9)
ax.bar(x + width, p03,  width, label="P(0-3)",    color="#F44336", alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(stage2_teams, rotation=40, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title(f"3.1  Stage 2 Marginal Outcome Probabilities  (n={int(hists[0].sum()):,})")
ax.set_ylabel("Probability")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Optimal pick'em card
from matplotlib.patches import FancyBboxPatch

_bg      = "#0d1117"
_panel_c = {"3-0": "#238636", "adv": "#1f6feb", "0-3": "#da3633"}

picks_30  = sorted(best_picks["three_zero"], key=lambda t: p30[stage2_teams.index(t)],  reverse=True)
picks_adv = sorted(best_picks["advance"],    key=lambda t: padv[stage2_teams.index(t)], reverse=True)
picks_03  = sorted(best_picks["zero_three"], key=lambda t: p03[stage2_teams.index(t)],  reverse=True)

panels = [
    ("3-0 PICKS",     picks_30,  p30,  "3-0", "#2ea043"),
    ("ADVANCE PICKS", picks_adv, padv, "adv", "#388bfd"),
    ("0-3 PICKS",     picks_03,  p03,  "0-3", "#f85149"),
]

fig = plt.figure(figsize=(15, 6), facecolor=_bg)
fig.subplots_adjust(left=0.02, right=0.98, top=0.78, bottom=0.05, wspace=0.06)
gs   = fig.add_gridspec(1, 3)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

for ax, (title, teams, probs, slot_key, bar_color) in zip(axes, panels):
    ax.set_facecolor(_bg)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(teams) - 0.5 + 1.2)
    ax.add_patch(FancyBboxPatch((0, len(teams) - 0.5 + 0.55), 1, 0.6,
                                boxstyle="round,pad=0.02", linewidth=0,
                                facecolor=_panel_c[slot_key], transform=ax.transData, clip_on=False))
    ax.text(0.5, len(teams) - 0.5 + 0.85, title,
            ha="center", va="center", fontsize=11, fontweight="bold",
            color="white", transform=ax.transData)
    for row, team in enumerate(reversed(teams)):
        idx = stage2_teams.index(team)
        p   = probs[idx]
        ax.barh(row, p, height=0.55, color=bar_color, alpha=0.85, zorder=2)
        ax.barh(row, 1, height=0.55, color="#21262d", alpha=0.6,  zorder=1)
        ax.text(0.02, row, team, va="center", ha="left", fontsize=9.5,
                color="white", fontweight="bold", zorder=3)
        ax.text(min(p + 0.03, 0.97), row, f"{p:.0%}", va="center", ha="left",
                fontsize=9, color="#8b949e", zorder=3)

n_sims = int(hists[0].sum())
e_cor  = sum(k * v for k, v in pick_dist.items())
p_ge5  = sum(v for k, v in pick_dist.items() if k >= 5)
fig.text(0.5, 0.92, "OPTIMAL PICK'EM — IEM Cologne 2026 Stage 2",
         ha="center", fontsize=14, fontweight="bold", color="white")
fig.text(0.5, 0.86, f"P(≥5 correct) = {p_ge5:.1%}   |   E[correct] = {e_cor:.2f}   |   n = {n_sims:,}",
         ha="center", fontsize=10, color="#8b949e")
plt.show()

In [ ]:
# 3.3 Score distribution
ks   = list(range(11))
vals = [pick_dist[k] for k in ks]

bar_colors = [
    "#F44336" if k < 4 else
    "#FF9800" if k < 5 else
    "#4CAF50"
    for k in ks
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(ks, vals, color=bar_colors, edgecolor="white", linewidth=0.6)
ax.bar_label(bars, fmt=lambda v: f"{v:.1%}" if v > 0.005 else "", padding=3, fontsize=9)
ax.axvline(4.5, color="orange", linestyle="--", linewidth=1.2, label=">=5 threshold")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xticks(ks)
ax.set_xlabel("Correct picks out of 10")
ax.set_ylabel("Probability")
ax.set_title(
    "3.3  Score Distribution — Optimal Pick Set\n"
    f"P(>=5) = {sum(v for k,v in pick_dist.items() if k>=5):.1%}  |  "
    f"E[correct] = {sum(k*v for k,v in pick_dist.items()):.2f}"
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Swiss outcome distribution — stacked bar per team
p_adv_only = padv - p30
p_elim     = 1 - padv - p03

order        = np.argsort(padv)[::-1]
teams_sorted = [stage2_teams[i] for i in order]
outcome_data = {
    "3-0":               p30[order],
    "Advance (3-1/3-2)": p_adv_only[order],
    "Elim (1-3/2-3)":    p_elim[order],
    "0-3":               p03[order],
}
bar_colors = ["#2ea043", "#388bfd", "#f0883e", "#f85149"]

x       = np.arange(len(teams_sorted))
fig, ax = plt.subplots(figsize=(16, 5))
bottoms = np.zeros(len(teams_sorted))
for (label, vals), color in zip(outcome_data.items(), bar_colors):
    ax.bar(x, vals, bottom=bottoms, label=label, color=color, alpha=0.9, width=0.78)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(teams_sorted, rotation=40, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title(f"3.4  Stage 2 Swiss Outcome Distribution  (n={int(hists[0].sum()):,})")
ax.set_ylabel("Probability")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Monte Carlo convergence
sims_hist = [h["sims"]    for h in conv_hist]
p5_hist   = [h["best_p5"] for h in conv_hist]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sims_hist, p5_hist, color="#388bfd", linewidth=2, marker="o", markersize=5, zorder=3)
ax.fill_between(sims_hist, p5_hist, alpha=0.12, color="#388bfd")
ax.axhline(p5_hist[-1], color="#8b949e", linestyle="--", linewidth=0.9,
           label=f"Final P(≥5) = {p5_hist[-1]:.1%}")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{int(v/1e3)}k"
))
ax.set_title("3.5  Monte Carlo Convergence — Best P(≥5 correct) vs Simulations Run")
ax.set_xlabel("Simulations run")
ax.set_ylabel("P(≥5 correct)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Pick stability heatmap
final_30  = set(conv_hist[-1]["picks_30"])
final_adv = set(conv_hist[-1]["picks_adv"])
final_03  = set(conv_hist[-1]["picks_03"])

slot_labels = ["3-0 #1", "3-0 #2"] + [f"Adv #{k}" for k in range(1, 7)] + ["0-3 #1", "0-3 #2"]
n_slots  = len(slot_labels)
n_rounds = len(conv_hist)

grid_teams  = np.empty((n_rounds, n_slots), dtype=object)
grid_stable = np.zeros((n_rounds, n_slots), dtype=bool)

for r, snap in enumerate(conv_hist):
    slot = 0
    for t in sorted(snap["picks_30"]):
        grid_teams[r, slot]  = t
        grid_stable[r, slot] = t in final_30
        slot += 1
    for t in sorted(snap["picks_adv"]):
        grid_teams[r, slot]  = t
        grid_stable[r, slot] = t in final_adv
        slot += 1
    for t in sorted(snap["picks_03"]):
        grid_teams[r, slot]  = t
        grid_stable[r, slot] = t in final_03
        slot += 1

fig, ax = plt.subplots(figsize=(16, max(3.5, n_rounds * 0.7 + 1.5)))
ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")
for spine in ax.spines.values():
    spine.set_edgecolor("#30363d")

for r in range(n_rounds):
    for c in range(n_slots):
        team  = grid_teams[r, c]
        color = "#2ea043" if grid_stable[r, c] else "#f85149"
        ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=color, alpha=0.75,
                                    edgecolor="#0d1117", linewidth=1.5))
        if team:
            ax.text(c + 0.5, r + 0.5, team[:9], ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold")

sims_labels = [
    f"{h['sims']/1e6:.2f}M" if h["sims"] >= 1e6 else f"{h['sims']/1e3:.0f}k"
    for h in conv_hist
]
ax.set_xlim(0, n_slots); ax.set_ylim(0, n_rounds)
ax.set_xticks(np.arange(n_slots) + 0.5)
ax.set_xticklabels(slot_labels, rotation=30, ha="right", fontsize=9, color="white")
ax.set_yticks(np.arange(n_rounds) + 0.5)
ax.set_yticklabels(sims_labels, fontsize=8, color="white")
ax.tick_params(colors="white")
ax.set_xlabel("Pick slot", color="white")
ax.set_ylabel("Sims run", color="white")
ax.set_title("3.6  Pick Stability  —  green = matches final pick · red = later changed",
             color="white", pad=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()